# 🌸 MBC 붓꽃 데이터 분석 · 2026.11.09
**MBC AI 데이터 저널리즘 수업 | 김형석**

- 원본 데이터: Kaggle [Iris Species](https://www.kaggle.com/datasets/uciml/iris)의 `Iris.csv`
- 목표: 원본 구조 확인 → 품질 점검 → 조건 검색·그룹 통계 → 차트 3종 → PNG 저장 → 질문·해석
- **주의:** 실행 전 `data/Iris.csv`를 준비하세요. 원본 CSV 파일은 제출받은 노트북/PNG 업로드에는 포함되지 않았습니다.
- 원본 실습 노트북을 제출용으로 정리한 버전이며, **이 정리본의 전체 실행은 아직 검증되지 않았습니다.**

## 1. 환경 준비 · CSV 불러오기

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Colab이면 Google Drive를 연결합니다. 로컬 VS Code에서는 건너뜁니다.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

# 프로젝트 폴더 / 원본 수업 폴더 중 존재하는 데이터 위치 선택
candidates = [
    Path('data/Iris.csv'),
    Path('Iris.csv'),
    Path('/content/drive/MyDrive/Colab Notebooks/Data_Analysis/MBC_Data_Visualization_20261109/data/Iris.csv'),
]
file_path = next((p for p in candidates if p.is_file()), None)
if file_path is None:
    raise FileNotFoundError('Iris.csv를 프로젝트 data/ 폴더에 넣거나 Colab Drive 경로를 확인하세요.')

iris = pd.read_csv(file_path)
required = {'Id','SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm','Species'}
missing = required - set(iris.columns)
if missing:
    raise ValueError(f'필요한 열이 없습니다: {sorted(missing)}')

columns = ['SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm']

# Colab에 한글 폰트 설치가 필요할 때만 실행; 실패해도 분석은 진행
font_path = Path('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
if not font_path.exists():
    try:
        import subprocess
        subprocess.run(['apt-get','-qq','install','-y','fonts-nanum'],check=False,
                       stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    except (OSError,FileNotFoundError):
        pass
if font_path.exists():
    fm.fontManager.addfont(str(font_path))
    plt.rcParams['font.family'] = 'NanumGothic'
else:
    print('안내: 한글 글꼴이 없으면 그래프 한글이 깨질 수 있습니다.')
plt.rcParams['axes.unicode_minus'] = False

# chart_dir 변수는 모든 저장 코드보다 먼저 정의되어야 합니다.
chart_dir = Path('charts')
# Colab에서 원본 Drive 작업 폴더로 읽었다면 같은 Drive의 charts에 저장
if '/content/drive/' in str(file_path):
    chart_dir = file_path.parent.parent / 'charts'
chart_dir.mkdir(parents=True, exist_ok=True)

print('원본 파일:',file_path)
print('데이터 크기:',iris.shape)
display(iris.head())

## 2. 데이터 구조 · 품질 검증

In [ ]:
print('열 이름:', iris.columns.tolist())
print('품종별 개수:')
print(iris['Species'].value_counts())
print('결측값:')
print(iris.isnull().sum())
print('Id 포함 중복 행:', iris.duplicated().sum())
print('Id 제외 동일 측정값·품종:', iris.duplicated(subset=columns+['Species']).sum())
# 측정값이 같아도 실제 다른 개체일 수 있으므로 자동 제거하지 않습니다.
print('기본 통계:')
display(iris[columns].describe())

## 3. 교재 형식으로 정리 · 조건 검색 · 그룹별 통계

In [ ]:
iris_book = iris.drop(columns=['Id']).rename(columns={
    'SepalLengthCm':'SepalLength', 'SepalWidthCm':'SepalWidth',
    'PetalLengthCm':'PetalLength', 'PetalWidthCm':'PetalWidth'
})
display(iris_book.head())
result = (iris['SepalLengthCm'] == 5.8) & (iris['PetalWidthCm'] == 1.9)
print('조건 검색 결과:')
display(iris.loc[result, :])
print('품종별 측정값 평균 (cm):')
display(iris.groupby('Species')[columns].mean().round(3))
print('품종별 측정값 합계 (cm):')
display(iris.groupby('Species')[columns].sum().round(2))

## 4. 그래프 ① 막대그래프 (Bar Chart)

**질문:** 세 가지 붓꽃 품종 중 평균 꽃잎 길이가 가장 긴 품종은?

**기존 노트북의 분석:** Iris-virginica가 평균 **5.55cm**로 가장 길었고, Iris-setosa **1.46cm**의 약 **3.8배**였다.

In [ ]:
avg = iris.groupby('Species')['PetalLengthCm'].mean()
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#B0B7C3' if name != avg.idxmax() else '#F28C28' for name in avg.index]
bars = ax.bar(avg.index, avg.values, color=colors)
ax.bar_label(bars, fmt='%.2f', padding=4)
ax.set_title('붓꽃 품종별 평균 꽃잎 길이 비교')
ax.set_xlabel('붓꽃 품종')
ax.set_ylabel('평균 꽃잎 길이 (cm)')
ax.set_ylim(0, max(6, avg.max()*1.08))
fig.text(0.1, 0.01,'출처: Kaggle Iris Species | 표본: 150개 | 단위: cm', fontsize=9, color='gray')
fig.tight_layout(rect=[0,0.04,1,1])
fig.savefig(chart_dir/'01_iris_bar_chart.png',dpi=200,bbox_inches='tight')
plt.show()
print(avg.round(2))

## 5. 그래프 ② 히스토그램 (Histogram)

**질문:** 네 가지 측정값은 어느 구간에 많이 분포하는가?

**기존 차트 해석:** 꽃받침 길이는 대체로 **5~6cm** 부근에, 꽃잎 너비는 상대적으로 작은 값에 많이 분포한다.

*참고: 네 측정값을 반투명으로 겹쳐 표현하므로 일부 분포가 가려질 수 있다. `bins=15`는 구간 수 설정이다.*

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
iris[columns].plot(kind='hist',bins=15,alpha=0.6,ax=ax)
ax.set_title('각 특징 값들의 빈도수 히스토그램')
ax.set_xlabel('길이 (cm)')
ax.set_ylabel('빈도수')
fig.text(0.1,0.01,'출처: Kaggle Iris Species | 표본: 150개 | 단위: cm',fontsize=9,color='gray')
fig.tight_layout(rect=[0,0.04,1,1])
fig.savefig(chart_dir/'02_iris_histogram.png',dpi=200,bbox_inches='tight')
plt.show()

## 6. 그래프 ③ 박스플롯 (Box Plot)

**질문:** 어떤 측정값의 분포가 넓고 이상치가 있는가?

**기존 차트 해석:** 꽃잎 길이의 분포가 넓으며, 꽃받침 너비에서 **상자수염 도표 기준의 이상치**가 관찰된다.

*참고: 박스플롯의 이상치가 데이터 오류라는 의미는 아니다.*

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
iris[columns].plot(kind='box', ax=ax)
ax.set_title('각 특징 값들의 분포와 이상치 박스플롯')
ax.set_xlabel('특징')
ax.set_ylabel('측정값 (cm)')
ax.tick_params(axis='x', labelrotation=15)
fig.text(0.1,0.01,'출처: Kaggle Iris Species | 표본: 150개 | 단위: cm',fontsize=9,color='gray')
fig.tight_layout(rect=[0,0.04,1,1])
fig.savefig(chart_dir/'03_iris_boxplot.png',dpi=200,bbox_inches='tight')
plt.show()

## 7. 저장 파일 확인 · 최종 검수

- 막대그래프는 **0에서 시작**하고 모든 막대 위에 **수치 라벨**을 표시한다.
- 막대그래프의 최고 평균값만 **주황색**으로 강조한다.
- 제목, 축 이름, 단위, 데이터 출처, 표본 크기를 확인한다.
- 이 분석은 품종별 **관찰된 차이**를 보여 주며 인과관계를 입증하지 않는다.

In [ ]:
for name in ['01_iris_bar_chart.png','02_iris_histogram.png','03_iris_boxplot.png']:
    p=chart_dir/name
    print(name, '저장됨' if p.is_file() else '없음', f'({p.stat().st_size:,} bytes)' if p.exists() else '')